In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. Učitavanje
# =========================
df = pd.read_csv("data\Motor vehicle insurance data (1).csv")

# =========================
# 2. Parsiranje datuma
# =========================
date_cols = [
    "Date_start_contract",
    "Date_last_renewal",
    "Date_next_renewal",
    "Date_birth",
    "Date_driving_licence",
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

# =========================
# 3. Sortiranje unutar ID
# =========================
df = df.sort_values(["ID", "Date_start_contract"])

# =========================
# 4. Flagovi: prvi / poslednji
# =========================
df["is_first"] = df.groupby("ID").cumcount() == 0
df["is_last"] = df.groupby("ID").cumcount(ascending=False) == 0

# =========================
# 5. Grupe I / II / III
# =========================
conditions = [
    (df["is_first"] & df["is_last"]),            # I
    (~df["is_first"] & df["is_last"]),           # II
    (~df["is_last"])                            # III
]

choices = ["I", "II", "III"]

df["group_type"] = np.select(conditions, choices)

# =========================
# 6. Split kandidati
# =========================
df_I = df[df["group_type"] == "I"]
df_II = df[df["group_type"] == "II"]
df_III = df[df["group_type"] == "III"]

# =========================
# 7. TEST = 50% I + 50% II (~20k)
# =========================
TARGET_TEST_SIZE = 20000
half = TARGET_TEST_SIZE // 2

df_I_sample = df_I.sample(n=min(half, len(df_I)), random_state=42)
df_II_sample = df_II.sample(n=min(half, len(df_II)), random_state=42)

test_df = pd.concat([df_I_sample, df_II_sample])

# =========================
# 8. TRAIN
# =========================
test_index = test_df.index

train_df = df.loc[~df.index.isin(test_index)]

# =========================
# 9. Provera
# =========================
print("TRAIN size:", len(train_df))
print("TEST size:", len(test_df))

print("\nTEST struktura:")
print(test_df["group_type"].value_counts())

print("\nTRAIN struktura:")
print(train_df["group_type"].value_counts())

KeyError: 'Date_start_contract'